In [1]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT

In [ ]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

# agent=agent_ssh
agent=agent_local
agent.Deploy()

In [10]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [11]:
# chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

Source(address='globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb', type=SourceType.GLOBUS)

In [12]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

kCvaS6w9
pprodigal
diamond


In [6]:
task.config = dict(
    nextflow = dict(
        preset = "slurm",
    ),
)

In [7]:
# agent.StageWorkflow(task, on_exist="clear")
agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-11_19-04-33  | connecting to deployed agent
2025-03-11_19-04-33  | starting relay service
  | > 2025-03-11_19-04-34  | connecting to relay as [sJnrHcVWbift]


 E| > 2025-03-11_19-04-34 E| relay server already running in [relay/connections]


2025-03-11_19-04-35  | sending metadata for workflow [kCvaS6w9]
2025-03-11_19-04-36  | staging
  | > including dev binds
  | > 2025-03-11_19-04-37  | api call to [stage_workflow] with [{'task_key': 'kCvaS6w9'}]
  | > 2025-03-11_19-04-37  | staging workflow [kCvaS6w9] with [4] given data instances
  | > 2025-03-11_19-04-37  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
  | > 2025-03-11_19-04-37  | work [/ws/runs/kCvaS6w9]
  | > 2025-03-11_19-04-37  | data [/msm_home/data]
  | > 2025-03-11_19-04-37  | external work [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/kCvaS6w9]
  | > 2025-03-11_19-04-37  | external data [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/data]
  | > 2025-03-11_19-04-37  | additional params:
  | > 2025-03-11_19-04-37  |     nextflow:
  | > 2025-03-11_19-04-37  |       preset: slurm
  | > 2025-03-11_19-04-37  | moving remote data libraries to [/msm_home/data]
  | > 2025-03-11_19-04-37  

In [8]:
import shutil
work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
    shutil.rmtree(work_root/p, ignore_errors=True)
shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

2025-03-11_19-07-59  | connecting to deployed agent
2025-03-11_19-07-59  | starting relay service


 E| > 2025-03-11_19-08-00 E| relay server already running in [relay/connections]


  | > 2025-03-11_19-08-00  | connecting to relay as [GihvqLwZv1Hk]
2025-03-11_19-08-01  | executing workflow
  | > including dev binds
  | > 2025-03-11_19-08-01  | api call to [execute_workflow] with [{'key': 'kCvaS6w9'}]
  | > 2025-03-11_19-08-01  | workspace [/msm_home/runs/kCvaS6w9]
  | > 2025-03-11_19-08-01  | external workspace [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/kCvaS6w9]
  | > 2025-03-11_19-08-01  | nextflow executable [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/lib/nextflow]
  | > 2025-03-11_19-08-01  | executing workflow [kCvaS6w9] with [2] steps
  | > 2025-03-11_19-08-01  | actualizing data if referencing remote sources
  | > 2025-03-11_19-08-01  | [D4Xf0lmlQIP7] is at [/msm_home/runs/kCvaS6w9/_metasmith/task/transforms/D4Xf0lmlQIP7]
  | > 2025-03-11_19-08-01  | [rrnqznjdBVnj] is at [/msm_home/runs/kCvaS6w9/_metasmith/task/data/rrnqznjdBVnj]
  | > 2025-03-11_19-08-01  | [2cOvJRRPrQor] is remote [globus://26024

In [9]:
# task = WorkflowTask(
#     plan=plan,
#     agent=agent,
#     data_libraries=[xgdb, refdb],
#     transform_libraries=[trlib],
#     config=dict(
#         nextflow=dict(
#             preset="default",
#             # slurm_account=slurm_account,
#         ),
#     ),
# )